# Week 13: Reinforcement Learning & RLHF
## Notebook 10: Twin Delayed DDPG (TD3): Overestimation Mitigation

### 1. Conceptual Intuition & Theoretical Background
In continuous action actor-critic methods like Deep Deterministic Policy Gradients (DDPG), Function approximation errors cause the critic to **consistently overestimate value functions**. This overestimation compounds across iterations, degrading policy performance.

**TD3 (Fujimoto, van Hoof, Meger, 2018) Three Pillars:**
1. **Clipped Double Q-Learning:**
   Maintains two independent critics ($Q_{\theta_1}, Q_{\theta_2}$) and uses the minimum value to construct Bellman targets:
   $$y = r + \gamma \min_{i=1, 2} Q_{\theta_i'}(s', \tilde{a})$$

2. **Target Policy Smoothing Regularization:**
   Adds clipped random noise to the target action to smooth out narrow Q-function peaks:
   $$\tilde{a} = \text{clip}\left( \pi_{\phi'}(s') + \text{clip}(\epsilon, -c, c),\ a_{\text{low}},\ a_{\text{high}} \right), \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

3. **Delayed Policy Updates:**
   Updates the actor network and target networks **less frequently** than the critic (e.g. 1 actor update per 2 critic updates).

---

### 2. Scratch Python Implementation: TD3 Action Smoothing & Target

In [ ]:
import numpy as np

def td3_target_action_smoothing(target_actions, noise_std=0.2, noise_clip=0.5, a_min=-1.0, a_max=1.0):
    """
    Applies clipped Gaussian noise to target actions for smoothing:
    a_tilde = clip(pi(s') + clip(N(0, sigma), -c, c), a_min, a_max)
    """
    noise = np.random.normal(0, noise_std, size=target_actions.shape)
    clipped_noise = np.clip(noise, -noise_clip, noise_clip)
    
    smoothed_actions = np.clip(target_actions + clipped_noise, a_min, a_max)
    return smoothed_actions

# Simulate Target Actions for 5 Batch Transitions
np.random.seed(42)
raw_actions = np.array([-0.8, -0.1, 0.0, 0.4, 0.95])

smoothed = td3_target_action_smoothing(raw_actions, noise_std=0.2, noise_clip=0.3)

print("--- TD3 Target Policy Action Smoothing ---")
print(f"Deterministic Target Actions:  {raw_actions}")
print(f"Stochastically Smoothed Actions: {np.round(smoothed, 3)}")
print("Smoothing prevents critics from exploiting narrow, non-smooth Q-function spikes!")

### 3. Step-by-Step Code Walkthrough
1. **Target Action Smoothing:** Perturbs target actions with bounded noise.
2. **Clipped Noise:** Restricts noise to $[-c, c]$ to preserve action semantics.
3. **Clipped Double-Q:** Takes the minimum of two critics to prevent overestimation bias.

---

### 4. Industry Applications & Trade-offs
- **Autonomous Drone Flight & Sim-to-Real:** Stable continuous trajectory planning under dynamic turbulence.
- **Trade-off:** Training two critic networks increases gradient computation overhead.

---

### 5. Top Interview Questions & Answers
**Q1: Why does DDPG suffer from overestimation bias?**
> **Answer:** Because the max operator or greedy policy update preferentially selects actions with overestimated Q-values. Over consecutive updates, this upward bias compounds.

**Q2: Why does TD3 delay actor updates relative to critic updates?**
> **Answer:** Updating the actor using an uncalibrated, rapidly shifting critic introduces instability. Delaying actor updates allows the critic to settle before guiding policy adjustments.